# Analysis of LST Subarray with CTLearn-Manager
We start by creating the CTLearn-Manager project for different two-telescope combinations in the LST subarray. Then, we train the stereo CTLearn models (stackedTRN) for each combination and monitor their training curves. The stereo CTLearn models are evaluated on a separate test set and the resulting DL2 files are merged into a single file per particle type and per telescope combination (also per pointing direction node, but we only consider one direction for now, which is the south pointing [zd 20°, az 180°]). Finally, we plot various DL2 control plots and produce the instrumental response functions (IRFs) and sensitivities to evaluate the performance.

An additional script (to be released in CTLearn) is used to calculate the weighted sum of the predictions from the different telescope combinations to obtain a final prediction for each event based on all triggered telescopes in the subarray fulfilling the default size cut of 50 photoelectrons. The verification of the final predictions is done by plotting various DL2 control plots and IRFs for the subarray using the combined predictions.

Note: Each cell can be run independently. However, the cells should be executed in the order they appear in the notebook to ensure that all dependencies are met. 

## 🧠 Create new `CTLearn-Manager` project

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject, DataSample
# Where all the models are stored
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Training samples, as many as you have directions and particle types, here we have 2 particle types and one direction
training_samples = [
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/train/gamma-diffuse/',
               pattern="gamma*.h5"),
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/train/proton/',
               pattern="proton*.h5"),
    ]
# Loop over telescope combinations and reconstruction tasks to create the model index file
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # General parameters
    tri_model_parameters = {
        "tri_model_nickname": f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo",
                                           #  for mono           for stereo
        'direction_reco' : "skydirection", # ["cameradirection", "skydirection"]
        "telescope_names": [f"LST-{tel_combo[0]}", f"LST-{tel_combo[1]}"],  # List of telescopes names
        "telescope_ids": [tel_combo[0], tel_combo[1]],  # List of telescope ids
        "max_training_epochs": 25,  # Can be changed later for training.
        "training_samples": training_samples,
        "stereo": True,  # True if stereo reconstruction, False if mono.
        'channels' : ['cleaned_image', 'cleaned_relative_peak_time'], # Order matters. # Default is ['cleaned_image', 'cleaned_relative_peak_time']
        'min_telescopes' : 2, # Minimum number of triggered telescopes for each events to be used in the model, if >=2, model will be stereo.
        'notes' : f"LST{tel_combo[0]}LST{tel_combo[1]}-CurCam stereo with default size cut of 50 pe",
    }
    # Create the tri model
    manager_project.create_tri_model(
            tri_model_parameters,
            overwrite=True
        )

## 🚀 Launch training


In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
from ctlearn_manager.utils import ClusterConfiguration
# Set up cluster configuration
cluster_configuration = ClusterConfiguration(environment="ctlearn", account="cta04", nodes=1, time='12:00:00', memory_mb=128000)
cluster_configuration.info()
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Loop over telescope combinations and launch training for each reconstruction task
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo", cluster_configuration=cluster_configuration)
    # Launch training for direction, energy, and type models
    tri_model.direction_model.launch_training(n_epochs=10, batch_size=32, save_best_validation_only=True)        
    tri_model.energy_model.launch_training(n_epochs=10, batch_size=32, save_best_validation_only=True)       
    tri_model.type_model.launch_training(n_epochs=6, batch_size=32, save_best_validation_only=True)     
    

## 📉 Load models and plot losses

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Loop over telescope combinations and plot the training loss for each reconstruction task
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo")
    # Plot training loss for direction, energy, and type models
    tri_model.plot_loss()

## 🧪 Launch testing

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject, DataSample
from ctlearn_manager.utils import ClusterConfiguration, ParticleType
import astropy.units as u
# Add electrons, gamma_diffuse and whatever you like to test your model on, each in a separate DataSample
testing_samples = [
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/test/gamma/',
               pattern="gamma*.h5"),
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/test/gamma-diffuse/',
               pattern="gamma*.h5"),
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/test/proton/',
               pattern="proton*.h5"),
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/test/electron/',
               pattern="electron*.h5"),
]
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set up cluster configuration
cluster_configuration = ClusterConfiguration(environment="ctlearn", account="cta04", nodes=1, time='30:00', memory_mb=128000)
# Loop over telescope combinations and launch testing for each reconstruction task
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo", cluster_configuration=cluster_configuration)
    # Set the testing data
    tri_model.set_testing_data(testing_samples)
    # See available testing directions and check the info of the cluster configuration
    tri_model.get_available_testing_directions()
    tri_model.cluster_configuration.info()
    # Launch testing for direction, energy, and type models with multiple particle types
    tri_model.launch_testing(
        20.0 * u.deg, 180.0 * u.deg,
        launch_particle_types=[
            ParticleType.GAMMA_POINT,
            ParticleType.GAMMA_DIFFUSE,
            ParticleType.PROTON,
            ParticleType.ELECTRON
        ],  # Add as much as you want
        batch_size=64,
        overwrite=True,
        config=f"/capstor/scratch/cscs/tmiener/configs/LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo_predictions.yml"
    )
    # The config files are stored in the 'src/ctlearn_manager/resources/configs/' directory
    # of the CTLearn-Manager repository. CSCS note: Remember to copy them to the scratch
    # of the CSCS cluster in order to use them on the worker node.

## 🔀 Merge the DL2 files
This is required for the IRF production, even if you already have one file.

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
from ctlearn_manager.utils import ParticleType
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Loop over telescope combinations and merge DL2 files for each particle type
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo")
    # Merge DL2 files for different particle types
    tri_model.merge_DL2_files(
        zenith, azimuth,
        particle_type=ParticleType.GAMMA_POINT,
        overwrite=True,
    )
    tri_model.merge_DL2_files(
        zenith, azimuth,
        particle_type=ParticleType.GAMMA_DIFFUSE,
        overwrite=True,
    )
    tri_model.merge_DL2_files(
        zenith, azimuth,
        particle_type=ParticleType.PROTON,
        overwrite=True,
    )
    tri_model.merge_DL2_files(
        zenith, azimuth,
        particle_type=ParticleType.ELECTRON,
        overwrite=True,
    )

## 📡 Produce the IRFs

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Loop over telescope combinations and produce various IRFs
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo")
    # Loop over different IRF types
    for irf_type in ["eff40", "eff70", "eff90", "opt"]:
        # Get the right config file
        config = f"/capstor/scratch/cscs/tmiener/configs/public-conf_{irf_type}.yml"
        # Produce the IRFs
        tri_model.produce_irfs(
            zenith, azimuth,
            config=config,
            pointlike=True,
            electrons=True,
            overwrite=True,
        )
        # For the optimal cuts, produce also the uncertainties
        # This might take a while!
        if irf_type == "opt":
            tri_model.produce_irfs_with_uncertainties(
                zenith, azimuth,
                config=config,
                pointlike=True,
                electrons=True,
                resume_file_index=None,
                resume_cut_index=None,
                overwrite=True,
            )

## 📉 Verify the DL2 control plots

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
from ctlearn_manager.utils import ParticleType, DefaultCuts, Cuts
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Loop over telescope combinations and plot various DL2 control plots
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo")
    # Plot various DL2 control plots
    tri_model.plot_ROC_curve_DL2(zenith, azimuth, 5)
    tri_model.plot_angular_resolution_DL2(
        [zenith.value] * u.deg, [azimuth.value] * u.deg, cuts=[DefaultCuts.EFF_40.value, DefaultCuts.EFF_70.value, DefaultCuts.EFF_90.value]
    )
    tri_model.plot_energy_resolution_DL2(
        [zenith.value] * u.deg, [azimuth.value] * u.deg, cuts=[DefaultCuts.EFF_40.value, DefaultCuts.EFF_70.value, DefaultCuts.EFF_90.value]
    )
    tri_model.plot_migration_matrix(
        zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT]
    )
    tri_model.plot_migration_matrix(
        zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT], cuts=Cuts(gammaness_cut=0.9)
    )
    tri_model.plot_DL2_AltAz(
        zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT]
    )
    tri_model.plot_DL2_AltAz(
        zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT], cuts=Cuts(gammaness_cut=0.9)
    )
    tri_model.plot_DL2_classification(
        zenith,
        azimuth,
        particle_types=[
            ParticleType.GAMMA_POINT,
            ParticleType.PROTON,
            ParticleType.ELECTRON,
        ],
    )
    tri_model.plot_DL2_energy(zenith, azimuth)

## 📉 Plot the IRFs

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Loop over telescope combinations and plot the IRFs
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
for tel_combo in tel_combination:
    # Open the corresponding CTLearn tri model
    tri_model = manager_project.open_tri_model(f"LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo")
    # Plot various IRFs and sensitivity benchmarks
    tri_model.plot_sensitivity_benchmark(zenith, azimuth)
    tri_model.plot_angular_resolution_benchmark(zenith, azimuth)
    tri_model.plot_energy_resolution_benchmark(zenith, azimuth)
    tri_model.plot_energy_bias_benchmark(zenith, azimuth)
    tri_model.plot_irfs(zenith, azimuth)

## Add LST Subarray combined predictions
Create model entry in the existing CTLearn-Manager project for the LSTSubarray combined predictions.

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject, DataSample
# Load the existing CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Same training samples, as many as you have directions and particle types, here we have 2 particle types and one direction
training_samples = [
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/train/gamma-diffuse/',
               pattern="gamma*.h5"),
    DataSample(directory='/capstor/scratch/cscs/tmiener/datasets/LSTSubarray/train/proton/',
               pattern="proton*.h5"),
    ]
# General parameters now for the 4LSTs stereo model
tri_model_parameters = {
    "tri_model_nickname": "4LSTs_CurCam_stereo",
                                        #  for mono           for stereo
    'direction_reco' : "skydirection", # ["cameradirection", "skydirection"]
    "telescope_names": ["LST-1", "LST-2", "LST-3", "LST-4"],  # List of telescopes names
    "telescope_ids": [1, 2, 3, 4],  # List of telescope ids
    "max_training_epochs": 25,  # Can be changed later for training.
    "training_samples": training_samples,
    "stereo": True,  # True if stereo reconstruction, False if mono.
    'channels' : ['cleaned_image', 'cleaned_relative_peak_time'], # Order matters. # Default is ['cleaned_image', 'cleaned_relative_peak_time']
    'min_telescopes' : 2, # Minimum number of triggered telescopes for each events to be used in the model, if >=2, model will be stereo.
    'notes' : "4LSTs-CurCam stereo with a weighted sum (based on hillas intensity sizes) of the two telescopes combinations predictions",
}
# Create the tri model
manager_project.create_tri_model(
        tri_model_parameters,
        overwrite=True
    )

## 🔀 Combine DL2 predictions from different two-telescope combinations

In [ ]:
# Import necessary libraries and modules
import os
# Set the project directory
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
# Define DL2 filenames for each particle type
# TODO: Currently hardcoded for simplicity, but can be automated later!
filenames = {
    "gamma_point": [ "gamma_runs1-100.dl2.h5", "gamma_runs101-200.dl2.h5", "gamma_runs201-300.dl2.h5",
                    "gamma_runs301-400.dl2.h5", "gamma_runs401-500.dl2.h5", "gamma_runs501-600.dl2.h5",
                    "gamma_runs601-700.dl2.h5", "gamma_runs701-800.dl2.h5", "gamma_runs801-900.dl2.h5",
                    "gamma_runs901-1000.dl2.h5"],
    "gamma_diffuse": ["gamma_runs1001-1200.dl2.h5", "gamma_runs1201-1400.dl2.h5", "gamma_runs1401-1600.dl2.h5",
                      "gamma_runs1601-1800.dl2.h5", "gamma_runs1801-2000.dl2.h5"],
    "electron": ["electron_runs1-200.dl2.h5", "electron_runs201-400.dl2.h5", "electron_runs401-600.dl2.h5",
                "electron_runs601-800.dl2.h5", "electron_runs801-1000.dl2.h5", "electron_runs1001-1200.dl2.h5",
                "electron_runs1201-1400.dl2.h5", "electron_runs1401-1600.dl2.h5", "electron_runs1601-1800.dl2.h5",
                "electron_runs1801-2000.dl2.h5"],
    "proton": ["proton_runs4001-4500.dl2.h5", "proton_runs4501-5000.dl2.h5"],
}
# Define telescope combinations
tel_combination = [[1,4], [4,3], [3,2], [2,1], [1,3], [4,2]]
# Set overwrite flag and output directory
overwrite = True
output_directory = f"{project_directory}/DL2/MC/4LSTs_CurCam_stereo"
# Loop over particle types and merge DL2 files
for particle_type in ["gamma_point", "proton", "electron", "gamma_diffuse"]:
    # Create output directory
    os.system(f"mkdir -p {output_directory}/{particle_type}/20.000_180.000")
    # Ensemble the command for combining DL2 predictions from different two-telescope combinations
    # for each unmerged DL2 filename.
    for filename in filenames[particle_type]:
        # Ensemble the command for combining DL2 predictions
        cmd = "python /users/tmiener/software/ctlearn/scripts/merge_subarray_table.py "
        for tel_combo in tel_combination:
            cmd += f" {project_directory}/DL2/MC/LST{tel_combo[0]}LST{tel_combo[1]}_CurCam_stereo/{particle_type}/20.000_180.000/{filename}"
        cmd += f" -o {output_directory}/{particle_type}/20.000_180.000/{filename} -v"
        # Remove existing file if overwrite is True
        if overwrite:
            os.system(f"rm -rf {output_directory}/{particle_type}/20.000_180.000/{filename}")
        # Print and execute the command
        print(cmd)
        os.system(cmd)

## 🔀 Merge DL2 files from combined predictions

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
from ctlearn_manager.utils import ParticleType
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Open the 4LSTs stereo tri model
tri_model = manager_project.open_tri_model("4LSTs_CurCam_stereo")
# Merge DL2 files for different particle types
tri_model.merge_DL2_files(
    zenith, azimuth,
    particle_type=ParticleType.GAMMA_POINT,
    overwrite=True,
)
tri_model.merge_DL2_files(
    zenith, azimuth,
    particle_type=ParticleType.GAMMA_DIFFUSE,
    overwrite=True,
)
tri_model.merge_DL2_files(
    zenith, azimuth,
    particle_type=ParticleType.PROTON,
    overwrite=True,
)
tri_model.merge_DL2_files(
    zenith, azimuth,
    particle_type=ParticleType.ELECTRON,
    overwrite=True,
)

## 📡 Produce the IRFs

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Open the corresponding CTLearn tri model
tri_model = manager_project.open_tri_model(f"4LSTs_CurCam_stereo")
# Loop over different IRF types
for irf_type in ["eff40", "eff70", "eff90", "opt"]:
    # Get the right config file
    config = f"/capstor/scratch/cscs/tmiener/configs/public-conf_{irf_type}.yml"
    # Produce the IRFs
    tri_model.produce_irfs(
        zenith, azimuth,
        config=config,
        pointlike=True,
        electrons=True,
        overwrite=True,
    )
    # For the optimal cuts, produce also the uncertainties
    # This might take a while!
    if irf_type == "opt":
        tri_model.produce_irfs_with_uncertainties(
            zenith, azimuth,
            config=config,
            pointlike=True,
            electrons=True,
            resume_file_index=None,
            resume_cut_index=None,
            overwrite=True,
        )

## 📉 Verify the DL2 control plots

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
from ctlearn_manager.utils import ParticleType, DefaultCuts, Cuts
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Open the corresponding CTLearn tri model
tri_model = manager_project.open_tri_model(f"4LSTs_CurCam_stereo")
# Plot various DL2 control plots
tri_model.plot_ROC_curve_DL2(zenith, azimuth, 5)
tri_model.plot_angular_resolution_DL2(
    [zenith.value] * u.deg, [azimuth.value] * u.deg, cuts=[DefaultCuts.EFF_40.value, DefaultCuts.EFF_70.value, DefaultCuts.EFF_90.value]
)
tri_model.plot_energy_resolution_DL2(
    [zenith.value] * u.deg, [azimuth.value] * u.deg, cuts=[DefaultCuts.EFF_40.value, DefaultCuts.EFF_70.value, DefaultCuts.EFF_90.value]
)
tri_model.plot_migration_matrix(
    zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT]
)
tri_model.plot_migration_matrix(
    zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT], cuts=Cuts(gammaness_cut=0.9)
)
tri_model.plot_DL2_AltAz(
    zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT]
)
tri_model.plot_DL2_AltAz(
    zenith, azimuth, particle_types=[ParticleType.GAMMA_POINT], cuts=Cuts(gammaness_cut=0.9)
)
tri_model.plot_DL2_classification(
    zenith,
    azimuth,
    particle_types=[
        ParticleType.GAMMA_POINT,
        ParticleType.PROTON,
        ParticleType.ELECTRON,
    ],
)
tri_model.plot_DL2_energy(zenith, azimuth)

## 📉 Plot the IRFs

In [ ]:
# Import necessary libraries and modules
from ctlearn_manager import CTLearnManagerProject
import astropy.units as u
# Load the CTLearn-Manager project
project_directory = "/capstor/scratch/cscs/tmiener/LSTSubarrayProject/DefaultSizeCut"
manager_project = CTLearnManagerProject(project_directory)
# Set zenith and azimuth for benchmarking
zenith, azimuth = 20 * u.deg, 180 * u.deg
# Open the corresponding CTLearn tri model
tri_model = manager_project.open_tri_model(f"4LSTs_CurCam_stereo")
# Plot various IRFs and sensitivity benchmarks
tri_model.plot_sensitivity_benchmark(zenith, azimuth)
tri_model.plot_angular_resolution_benchmark(zenith, azimuth)
tri_model.plot_energy_resolution_benchmark(zenith, azimuth)
tri_model.plot_energy_bias_benchmark(zenith, azimuth)
tri_model.plot_irfs(zenith, azimuth)